In [1]:
import re
from pathlib import Path
import shutil

In [ ]:
import re

def est_ligne_parasite(ligne):
    ligne = ligne.strip()

    if not ligne:
        return False

    # Numéro de page isolé
    if re.match(r'^\d+$', ligne):
        return True

    # Chiffre romain isolé 
    if re.match(r'^[IVXLCDM]+\.?$', ligne, flags=re.IGNORECASE):
        return True

    # Titre de chapitre du type :
    if re.match(r'^[IVXLCDM]+(?:\.\s*|\s*[–-]\s*|\s+).+$', ligne):
        return True

    # Tables / listes éditoriales éventuelles
    if re.match(
        r'^(table des matières(?: du titre)?|liste(?: générale)? des titres|table ?)$',
        ligne,
        flags=re.IGNORECASE
    ):
        return True

    # Lignes entièrement en majuscules parasites
    if re.match(r"^[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸÆŒ0-9' -]{5,}$", ligne):
        return True

    return False


def supprimer_avant_premier_chapitre(texte):
    match = re.search(r'(?m)^\s*I\.?\s*$', texte)
    if match:
        return texte[match.start():]
    return texte


def nettoyer_texte(texte):
    # 0. On garde seulement le roman à partir du premier I
    texte = supprimer_avant_premier_chapitre(texte)

    # 1. Suppression des lignes parasites
    lignes = texte.splitlines()
    lignes_filtrees = [ligne for ligne in lignes if not est_ligne_parasite(ligne)]
    texte = "\n".join(lignes_filtrees)

    # 2. Normalisation légère
    texte = texte.replace("«", "")
    texte = texte.replace("»", "")
    texte = texte.replace('"', "")
    texte = texte.replace("—", " ")
    texte = texte.replace("–", " ")
    texte = texte.replace("<<", " ")
    texte = texte.replace(">>", " ")
    texte = texte.replace("<", " ")
    texte = texte.replace(">", " ")

    # 3. Réparer les mots coupés en fin de ligne
    # ex. appa-
    # raître -> apparaître
    texte = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', texte)

    # Supprimer les chiffres collés aux mots
    texte = re.sub(r'(?<=[A-Za-zÀ-ÖØ-öø-ÿÆŒæœ])\d+', '', texte)

    # Supprimer les espaces avant ponctuation
    texte = re.sub(r'\s+([.,;:?!])', r'\1', texte)

    # 4. Compactage
    texte = re.sub(r'\n+', '\n', texte)
    texte = re.sub(r'[ \t]+', ' ', texte).strip()
    texte = re.sub(r'\s+', ' ', texte).strip()

    # 5. Protection des points de suspension
    texte = texte.replace("...", "<ELLIPSIS>")
    texte = texte.replace("…", "<ELLIPSIS_CHAR>")

    # 6. Protection des abréviations
    abreviations = [
        "M.", "MM.", "Mr.", "Mme.", "Mmes.", "Mlle.",
        "Dr.", "Pr.", "St.", "Ste.", "Sr.", "Jr.",
        "etc.", "cf.", "chap.", "vol.", "t.", "1.", "2.", "3.", "4.",
        "5.", "6.", "7.", "8.", "9."
    ]

    for abbr in abreviations:
        texte = texte.replace(abbr, abbr.replace(".", "<ABBR_DOT>"))

    # 7. Découpage en phrases
    texte = re.sub(r'([.!?])\s+', r'\1\n', texte)

    # 8. Restaurations
    texte = texte.replace("<ABBR_DOT>", ".")
    texte = texte.replace("<ELLIPSIS>", "...")
    texte = texte.replace("<ELLIPSIS_CHAR>", "…")

    # 9. Nettoyage final
    texte = re.sub(r'\n{2,}', '\n', texte)
    lignes = [ligne.strip() for ligne in texte.split('\n') if ligne.strip()]
    texte = '\n'.join(lignes)

    return texte

In [11]:
dossier_source = Path("/Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute")

for source in dossier_source.glob("*.txt"):
    if source.stem.endswith("_clean"):
        continue

    destination = source.with_name(source.stem + "_clean.txt")

    try:
        with open(source, "r", encoding="utf-8") as f:
            contenu_brut = f.read()

        contenu_propre = nettoyer_texte(contenu_brut)

        with open(destination, "w", encoding="utf-8") as f:
            f.write(contenu_propre)

        print(f"Nettoyage terminé : {destination}")

    except FileNotFoundError:
        print(f"Fichier introuvable : {source}")

    except Exception as e:
        print(f"Erreur avec {source.name} : {e}")

Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1896_2_Rome._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1884_12_La_joie_de_vivre._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1867_Therese_Raquin._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1883_11_Au_Bonheur_des_dames._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1865_La_confession_de_Claude._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1890_17_La_bete_humaine._clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/zola_brute/1871_1_La_fortune_des_Rougon._clean.txt
Nettoyage terminé : 